## Setup
### Prerequisites for analysis *from raw data*
- Rust Cargo
    - `curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh`
- Python uv
    - `curl -LsSf https://astral.sh/uv/install.sh | sh`

#### Install Skope, fetch plotting scripts and reference genomes
```
RUSTFLAGS="-C target-cpu=native" cargo install --git https://github.com/bede/skope.git
git clone https://github.com/bede/zmrp
```

In [1]:
import os
import subprocess
import numpy as np
import pandas as pd
import altair as alt

from pathlib import Path

def run_cmd(cmd):
    subprocess.run(cmd, shell=True, check=True, text=True)
    
@alt.theme.register('larger_font_theme', enable=True)
def larger_font_theme():
    return alt.theme.ThemeConfig({
        'config': {
            'title': {
                'fontSize': 16
            },
            'axis': {
                'labelFontSize': 14,
                'titleFontSize': 16,
                'titlePadding': 15
            },
            'header': {
                'labelFontSize': 14,
                'titleFontSize': 18
            },
            'legend': {
                'labelFontSize': 14,
                'titleFontSize': 14
            }
        }
    })

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

### Metadata

In [2]:
runs = ['DS_NEBtagged_sequenase_Zepto_18_11_2025', 'MW_FRM_cDNA_NEB_RLB_04092025', 'DS_BRC-zepto_roundAB_2_02_2026']

samples_df = pd.read_csv("samples.tsv", sep="\t")
samples_df.head()

,run,barcode,condition,replicate,name,run_name,full_name
0,DS_NEBtagged_sequenase_Zepto_18_11_2025,barcode05,full/neb,1,full/neb/1,Round A/B,roundab/full/neb/1
1,DS_NEBtagged_sequenase_Zepto_18_11_2025,barcode06,full/neb,2,full/neb/2,Round A/B,roundab/full/neb/2
2,DS_NEBtagged_sequenase_Zepto_18_11_2025,barcode07,full/neb,neg,full/neb/neg,Round A/B,roundab/full/neb/neg
3,DS_NEBtagged_sequenase_Zepto_18_11_2025,barcode20,frag/neb,1,frag/neb/1,Round A/B,roundab/frag/neb/1
4,DS_NEBtagged_sequenase_Zepto_18_11_2025,barcode21,frag/neb,neg,frag/neb/neg,Round A/B,roundab/frag/neb/neg


In [3]:
zmrp_df = pd.read_csv("zmrp/genomes.tsv", sep="\t")
zmrp_df.head()

,name,full_name,category,genome,pool,segmented,status,accession
0,AdV-1,Adenovirus Type 1,dna-virus,dna,1,False,complete,AC_000017.1
1,AdV-3,Adenovirus Type 3,dna-virus,dna,1,False,complete,DQ099432.4
2,AdV-31,Adenovirus Type 31,dna-virus,dna,1,False,complete,AM749299.1
3,Flu-A-H1N1-NY,Influenza A H1N1 A/NY/02/2009,rna-virus,rna,1,True,complete,KT180555.1
4,Flu-A-H3N2,Influenza A H3N2 A/Brisbane/10/07,rna-virus,rna,1,True,complete,KJ609211.1


### Fetch fastqs

Uncomment if analysing from raw data

In [4]:
# S3_BUCKET = "s3://quick-research-protocol-optimisation/fastq"
# S3_ENDPOINT = "https://s3.climb.ac.uk"

# base_outdir = Path("fastq")

# for run, df_run in samples_df.groupby("run"):
#     outdir = base_outdir / run
#     outdir.mkdir(parents=True, exist_ok=True)

#     for barcode in df_run["barcode"].unique():
#         cmd = (
#             f"aws --endpoint-url={S3_ENDPOINT} --no-sign-request "
#             f"s3 cp {S3_BUCKET}/{run}/{barcode}.fastq.gz {outdir} --quiet"
#         )

#         result = subprocess.run(cmd, shell=True)
#         if result.returncode == 0:
#             print(f"Downloaded {run}/{barcode}.fastq.gz")

### Fetch chm13v2

Uncomment if analysing from raw data

In [5]:
# !wget -O assets/classify-index/human.fa.gz https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/analysis_set/chm13v2.0.fa.gz

In [6]:
# %%bash
# cp zmrp/zmrp21.rna-viruses.fa assets/classify-index/rna-viruses.fa
# cp zmrp/zmrp21.dna-viruses.fa assets/classify-index/dna-viruses.fa
# cp zmrp/zmrp21.bacteria.fa assets/classify-index/bacteria.fa
# skope index build assets/classify-index > assets/classify.idx

### Estimate composition and virus genome containment using Skope
Uncomment if analysing from raw data

In [7]:
# os.environ['PATH'] = f"{os.path.expanduser('~/.local/bin')}:{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"
# !mkdir -p figs fastq assets/classify-index

# df = samples_df

# names = ",".join(df["full_name"])

# fastqs = (
#     "fastq/"
#     + df["run"]
#     + "/"
#     + df["barcode"]
#     + ".fastq.gz"
# )

# run_cmd(f"skope classify -k 31 assets/coarse-index --threads 16 --limit 1G --names {names} {' '.join(fastqs)} > classified-coarse.tsv")
# run_cmd(f"skope classify -k 31 assets/fine-index --threads 16 --limit 1G --names {names} {' '.join(fastqs)} > classified-fine.tsv")
# run_cmd(f"skope classify -d -k 31 assets/fine-index --threads 16 --limit 1G --names {names} {' '.join(fastqs)} > classified-fine.dis.tsv")

# run_cmd(f"skope query --threads 16 --limit 1G --names {names} zmrp/zmrp21.viruses.fa {' '.join(fastqs)} > query.viruses.1G.tsv")

### Find highest-yielding replicates per condition
Where highest yielding means highest viral containment10

In [8]:
df = pd.read_csv("query.viruses.1G.tsv", sep="\t")
total = df.query("target == 'TOTAL'").copy()

total["replicate"] = total["sample"].str.split("/").str[-1]
total["condition"] = total["sample"].str.rsplit("/", n=1).str[0]

highest_yielding_df = total.loc[total.groupby("condition")["containment10"].idxmax()]
highest_yielding_df = highest_yielding_df.drop(columns=["replicate", "condition"])
highest_yielding_samples = highest_yielding_df["sample"].tolist()
highest_yielding_samples_core = [s for s in highest_yielding_samples if "/full/neb/" in s or "/frag/neb/" in s or "/frag/rlb/" in s or "/frag/neb+rlb/" in s]
highest_yielding_df

,target,sample,containment1,containment1_hits,containment10,containment10_hits,target_length,target_kmers,median_nz_abundance
67,TOTAL,roundab/frag/neb/1,0.53198,7262,0.48919,6678,0,13651,382
152,TOTAL,roundab/frag/neb+rlb/1,0.05846,798,0.00015,2,0,13651,1
101,TOTAL,roundab/frag/rlb/1,0.08981,1226,0.00007,1,0,13651,2
33,TOTAL,roundab/full/neb/2,0.66420,9067,0.62464,8527,0,13651,392
611,TOTAL,roundabspike/frag/neb/1,0.37653,5140,0.32869,4487,0,13651,787
764,TOTAL,roundabspike/frag/neb+rlb/2,0.03817,521,0.00022,3,0,13651,1
696,TOTAL,roundabspike/frag/rlb/2,0.02732,373,0.00322,44,0,13651,8
543,TOTAL,roundabspike/full/neb/1,0.42004,5734,0.37191,5077,0,13651,379
356,TOTAL,smart9n/frag/neb/2,0.87517,11947,0.82016,11196,0,13651,202
407,TOTAL,smart9n/frag/neb+rlb/1,0.43396,5924,0.00864,118,0,13651,2


In [9]:
highest_yielding_samples_core = sorted(highest_yielding_samples_core, key=lambda s: (
    0 if s.split('/')[0] == 'smart9n' else 1,
    0 if s.split('/')[1] == 'full' else 1,
    {'neb': 0, 'rlb': 1, 'neb+rlb': 2}.get(s.split('/')[2], 3),
    int(s.split('/')[3])
))
highest_yielding_samples_core

['smart9n/full/neb/1',
 'smart9n/frag/neb/2',
 'smart9n/frag/rlb/2',
 'smart9n/frag/neb+rlb/1',
 'roundabspike/full/neb/1',
 'roundab/full/neb/2',
 'roundab/frag/neb/1',
 'roundabspike/frag/neb/1',
 'roundab/frag/rlb/1',
 'roundabspike/frag/rlb/2',
 'roundab/frag/neb+rlb/1',
 'roundabspike/frag/neb+rlb/2']

## Composition

In [10]:
df = pd.read_csv("classified-coarse.tsv", sep="\t")
samples = highest_yielding_samples_core
group_order = [
    "rna-viruses",
    "dna-viruses",
    "bacteria",
    "human",
    "ambiguous",
    "unclassified",
]
df = df[df["sample"].isin(samples)]
group_rank = {g: i for i, g in enumerate(group_order)}
df["group_order"] = df["group"].map(group_rank)

# Extract prefix and short name
protocol_labels = {"smart9n": "SMART-9N", "roundab": "Round A/B"}
df["protocol"] = df["sample"].str.split("/").str[0].map(protocol_labels)
df["short_name"] = df["sample"].apply(lambda s: "/".join(s.split("/")[1:-1]))

df = df.query("protocol == 'SMART-9N'")

short_name_order = ["full/neb", "frag/neb", "frag/rlb", "frag/neb+rlb"]

# Pre-aggregate
df_agg = df.groupby(["sample", "protocol", "short_name", "group", "group_order"], as_index=False)["bases"].sum()

chart = alt.Chart(df_agg).mark_bar().encode(
    x=alt.X("short_name:N", axis=alt.Axis(labelAngle=-45), title="",
             sort=short_name_order),
    y=alt.Y("bases:Q", title="Bases",
             scale=alt.Scale(domain=[0, 1000000000])),
    color=alt.Color("group:N", title="",
                     sort=group_order),
    order=alt.Order("group_order:Q", sort="descending"),
).properties(
    width=200,
    height=200,
    title="SMART-9N yield by taxon (1Gbp)",
)

chart.save("taxonomic-abs-smart9n.png", scale_factor=2)
chart

alt.Chart(...)

In [11]:
df = pd.read_csv("classified-coarse.tsv", sep="\t")
samples = highest_yielding_samples_core
group_order = [
    "rna-viruses",
    "dna-viruses",
    "bacteria",
    "human",
    "ambiguous",
    "unclassified",
]
df = df[df["sample"].isin(samples)]
group_rank = {g: i for i, g in enumerate(group_order)}
df["group_order"] = df["group"].map(group_rank)

# Extract prefix and short name
protocol_labels = {"smart9n": "SMART-9N", "roundab": "Round A/B"}
df["protocol"] = df["sample"].str.split("/").str[0].map(protocol_labels)
df["short_name"] = df["sample"].apply(lambda s: "/".join(s.split("/")[1:-1]))

df = df.query("protocol == 'Round A/B'")

short_name_order = ["full/neb", "frag/neb", "frag/rlb", "frag/neb+rlb"]

# Pre-aggregate
df_agg = df.groupby(["sample", "protocol", "short_name", "group", "group_order"], as_index=False)["bases"].sum()

chart = alt.Chart(df_agg).mark_bar().encode(
    x=alt.X("short_name:N", axis=alt.Axis(labelAngle=-45), title="",
             sort=short_name_order),
    y=alt.Y("bases:Q", title="Bases",
             scale=alt.Scale(domain=[0, 1000000000])),
    color=alt.Color("group:N", title="",
                     sort=group_order),
    order=alt.Order("group_order:Q", sort="descending"),
).properties(
    width=200,
    height=200,
    title="Round A/B yield by taxon (1Gbp)",
)

chart.save("taxonomic-abs-roundab.png", scale_factor=2)
chart

alt.Chart(...)

## Containment vs. abundance

In [12]:
query_df = (
    pd.read_csv("query.viruses.1G.tsv", sep="\t")
    .query("sample in @highest_yielding_samples_core")
    .query("target != 'TOTAL'")
    .merge(zmrp_df[["name", "category", "genome"]], left_on="target", right_on="name")
    .drop(columns="name")
    .merge(samples_df[["run_name", "full_name", "condition", "replicate"]], left_on="sample", right_on="full_name")
    .drop(columns="full_name")
    .query("run_name == 'SMART-9N'")

)

col_order = ['full/neb', 'frag/neb', 'frag/rlb', 'frag/neb+rlb']

chart = (
    alt.Chart(query_df)
    .mark_circle(size=100)
    .encode(
        x=alt.X('containment1:Q', 
                title='Containment at depth ≥ 1', 
                scale=alt.Scale(domain=[0, 1])),
        y=alt.Y('median_nz_abundance:Q',
                scale=alt.Scale(type='log'),
                title='Median abundance',
        axis=alt.Axis(gridOpacity=0.3, tickCount=5)),  # Add this line
        color=alt.Color('genome:N', 
                        title='',
                        sort=['dna', 'rna']),
        tooltip=[
            alt.Tooltip('target:N', title='Target'),
            alt.Tooltip('sample:N', title=''),
            alt.Tooltip('protocol:N', title='Protocol'),
            alt.Tooltip('genome:N', title='Genome'),
            alt.Tooltip('containment1:Q', title='Containment', format='.2f'),
            alt.Tooltip('containment1_hits:Q', title='K-mer hits'),
            alt.Tooltip('median_nz_abundance:Q', title='Median abundance'),
            alt.Tooltip('total_syncmers:Q', title='Total syncmers', format=','),
        ]
    )
    .properties(width=140, height=140)
    .facet(
        column=alt.Column('condition:N', 
                          title='',
                          sort=col_order,
                          header=alt.Header(labelFontSize=12, titleFontSize=14)),
    )
    .properties(title='SMART-9N virus genome containment vs. median abundance (1Gbp)')
    .configure_axis(labelFontSize=12, titleFontSize=12)
)
chart.save("smart9n-1x4.png", scale_factor=2)
chart

alt.FacetChart(...)

In [13]:
query_df = (
    pd.read_csv("query.viruses.1G.tsv", sep="\t")
    .query("sample in @highest_yielding_samples_core")
    .query("target != 'TOTAL'")
    .merge(zmrp_df[["name", "category", "genome"]], left_on="target", right_on="name")
    .drop(columns="name")
    .merge(samples_df[["run_name", "full_name", "condition", "replicate"]], left_on="sample", right_on="full_name")
    .drop(columns="full_name")
    .query("run_name == 'Round A/B'")
    .query("median_nz_abundance > 0")

)

col_order = ['full/neb', 'frag/neb', 'frag/rlb', 'frag/neb+rlb']

chart = (
    alt.Chart(query_df)
    .mark_circle(size=100)
    .encode(
        x=alt.X('containment1:Q', 
                title='Containment at depth ≥ 1', 
                scale=alt.Scale(domain=[0, 1])),
        y=alt.Y('median_nz_abundance:Q',
                scale=alt.Scale(type='log'),
                title='Median abundance',
        axis=alt.Axis(gridOpacity=0.3, tickCount=5)),  # Add this line
        color=alt.Color('genome:N', 
                        title='',
                        sort=['dna', 'rna']),
        tooltip=[
            alt.Tooltip('target:N', title='Target'),
            alt.Tooltip('sample:N', title=''),
            alt.Tooltip('protocol:N', title='Protocol'),
            alt.Tooltip('genome:N', title='Genome'),
            alt.Tooltip('containment1:Q', title='Containment', format='.2f'),
            alt.Tooltip('containment1_hits:Q', title='K-mer hits'),
            alt.Tooltip('median_nz_abundance:Q', title='Median abundance'),
            alt.Tooltip('total_syncmers:Q', title='Total syncmers', format=','),
        ]
    )
    .properties(width=140, height=140)
    .facet(
        column=alt.Column('condition:N', 
                          title='',
                          sort=col_order,
                          header=alt.Header(labelFontSize=12, titleFontSize=14)),
    )
    .properties(title='Round A/B virus genome containment vs. median abundance (1Gbp)')
    .configure_axis(labelFontSize=12, titleFontSize=12)
)
chart.save("roundab-1x4.png", scale_factor=2)
chart

alt.FacetChart(...)

## Containment vs. abundance and log fold change between full/neb and frag/rlb

In [14]:
query_df = (
    pd.read_csv("query.viruses.1G.tsv", sep="\t")
    .query("target != 'TOTAL'")
    .merge(zmrp_df[["name", "category", "genome"]], left_on="target", right_on="name")
    .drop(columns="name")
    .merge(samples_df[["run_name", "full_name", "condition", "replicate"]], left_on="sample", right_on="full_name")
    .drop(columns="full_name")
    .query("condition == 'full/neb' or condition == 'frag/rlb'")
    .query("run_name == 'Round A/B (spike)' or sample == 'roundab/full/neb/2' or sample == 'roundab/frag/rlb/1'")
    .query("median_nz_abundance > 0")
    .query("replicate != 'neg'")
)
col_order = ['full/neb', 'frag/rlb']
spike_labels = {
    '1': 'low background', # BRC 130
    '2': 'high background', # BRC 156
    '3': 'medium background', # BRC 205
}
extra_labels = {
    'roundab/full/neb/2': 'unspiked',
    'roundab/frag/rlb/1': 'unspiked',
}
query_df['facet_label'] = query_df['sample'].map(extra_labels).fillna(query_df['replicate'].map(spike_labels))
fig2a = (
    alt.Chart(query_df)
    .mark_circle(size=100, color='black')
    .encode(
        x=alt.X('containment1:Q', 
                title='Containment at depth ≥ 1', 
                scale=alt.Scale(domain=[0, 1])),
        y=alt.Y('median_nz_abundance:Q',
                scale=alt.Scale(type='log', domain=[1, query_df['median_nz_abundance'].max() * 1.5]),
                title='Median abundance',
                axis=alt.Axis(gridOpacity=0.3, tickCount=5)),
    )
    .properties(width=130, height=130)
    .facet(
        row=alt.Column('condition:N', 
                       title='',
                       sort=col_order,
                       header=alt.Header(labelFontSize=12, titleFontSize=14)),
        column=alt.Row('facet_label:N',
                       title='',
                       sort=['unspiked', 'low background', 'medium background', 'high background'],
                       header=alt.Header(labelFontSize=12, titleFontSize=14))
    )
    .properties(title=alt.TitleParams(
        'Round A/B virus genome containment vs. median abundance (1Gbp)',
        anchor='middle'
    ))
)
# fig2a.save("figs/roundab-spiked-unspiked-2x4-noneg.png", scale_factor=2)
fig2a

sample_labels = {
    '1': 'low background',
    '2': 'high background',
    '3': 'medium background',
    'neg': 'Negative control'
}

facet_order = ['unspiked', 'low background', 'medium background', 'high background']

# Highest containment reps for each condition
reps = [s for s in highest_yielding_samples_core if "roundab/full/neb/" in s or "roundab/frag/rlb/" in s]

df = pd.concat([
    pd.read_csv('classified-fine.dis.tsv', sep='\t')
    .query("sample.str.startswith('roundabspike/')", engine='python')
    .assign(
        replicate=lambda x: x['sample'].str.split('/').str[-1],
        replicate_name=lambda x: x['replicate'].map(sample_labels)
    )
    .query("replicate != 'neg'"),
    pd.read_csv('classified-fine.dis.tsv', sep='\t')
    .query("sample in @reps")
    .assign(
        replicate_name ="unspiked" 
    )
]).drop(
    "replicate", axis=1
)


sample_labels = {
    '1': 'low background',
    '2': 'high background',
    '3': 'medium background',
    'neg': 'Negative control'
}
facet_order = ['unspiked', 'low background', 'medium background', 'high background']


full_neb = (
    df.query("sample.str.contains('/full/neb/')", engine='python')
    .set_index(['replicate_name', 'group'])
)
frag_rlb = (
    df.query("sample.str.contains('/frag/rlb/')", engine='python')
    .set_index(['replicate_name', 'group'])
)

virus_targets = [
    g for g in df['group'].unique()
    if g not in ('unclassified', 'ambiguous')
]

fc_rows = []
for (rep_name, virus) in full_neb.index:
    if virus not in virus_targets:
        continue
    if (rep_name, virus) not in frag_rlb.index:
        continue
    log_fc = (
        np.log10(full_neb.loc[(rep_name, virus), 'bases'] + 1)
        - np.log10(frag_rlb.loc[(rep_name, virus), 'bases'] + 1)
    )
    fc_rows.append({
        'replicate_name': rep_name,
        'virus': virus,
        'log_fc': log_fc
    })

fc_df = pd.DataFrame(fc_rows)

virus_order = (
    fc_df.groupby('virus')['log_fc']
    .max()
    .sort_values(ascending=False)
    .index
    .tolist()
)

bars = alt.Chart(fc_df).mark_bar().encode(
    x=alt.X(
        'virus:N',
        sort=virus_order,
        axis=alt.Axis(labelAngle=-45),
        title=None
    ),
    y=alt.Y(
        'log_fc:Q',
        title='log10 fold change in classified bp'
    ),
    color=alt.Color(
        'replicate_name:N',
        title='',
        sort=facet_order,
        legend=alt.Legend(
            orient='none',
            legendX=470,
            legendY=1,
            fillColor='white',
            # strokeColor='lightgray',
            padding=2
        )
    ),
    xOffset=alt.XOffset('replicate_name:N', sort=facet_order)
).properties(
    width=620,
    height=250,
)

zero_line = alt.Chart(
    pd.DataFrame({'y': [0]})
).mark_rule(
    strokeDash=[4, 4],
    color='gray'
).encode(y='y:Q')


fig2b = bars + zero_line
fig2b = fig2b.properties(title=alt.TitleParams(
    'Round A/B (spiked) log fold change in viral yield (frag/rlb vs full/neb)',
    anchor='middle'
))

combined = alt.vconcat(fig2a, fig2b, spacing=40).resolve_scale(color='independent').configure_axis(
    labelFontSize=12,
    titleFontSize=12
).configure_title(
    offset=12  # applies to all titles
)
combined.save("fig2.png", scale_factor=2)
combined

alt.VConcatChart(...)